# Survey of Consumer Finances (SCF)  
This notebook is going to be the first notebook in exploring the summarized version of the Survey of Consumer Finances. The summarized version is found on the website: https://www.federalreserve.gov/econres/scfindex.htm . We have decided to explore the summarized version first, as it has been cleaned and summarized by the Fed. They have actually created five different sets of the dataset with differing imputed values for missing values. We will need to pick one of the versions or potentially average the results from all five. Let's start by importing our libraries and downloading the dataset if don't yet have it.

## Libraries


In [ ]:
import requests
import zipfile
import io

import numpy as np
import pandas as pd
import altair as alt

# Deactivate the max rows and columns limit for Altair
alt.data_transformers.disable_max_rows()

url = "https://www.federalreserve.gov/econres/files/scfp2022excel.zip"

## Data Download or Import
Alright, now that we have all of this, I want to see if we have the file. If we have the file, we can just import it, however if we haven't downloaded it yet, we will go to the website and download it. 

In [ ]:
# Check to see if the data file already exists before attempting to download
try:
    print("Importing data from the Survey of Consumer Finances...")
    data = pd.read_csv("SCFP2022.csv")
    print("Data imported successfully.")

#If it doesn't exist we can download it form the Federal Reserve's website
except:
    print("File not found. Downloading survey data from the Federal Reserve's website...")
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers)
    print(f"Status code: {response.status_code}") # Check if the request was successful
    if response.status_code == 200:
        with zipfile.ZipFile(io.BytesIO(response.content)) as z:
            z.extractall()  # Extracts all files in the zip to the current directory
        print("Data downloaded and extracted successfully.")
        data = pd.read_csv("SCFP2022.csv")
    else:
        print("Failed to download data. Please check the URL and try again.")


## Inital Exploration
To start, we should just get a sense of the amount of data that we have in our dataframe. What missing values we have, the columns, we have, descriptive statistics etc. 

In [ ]:
print(f"Dataframe shape: {data.shape}")
data.head(10)

In [ ]:
columns = data.columns.tolist()
print(columns)

Alright, 357 columns is going to be way more than we need. So let's take a look at the documentation on the website to see which ones will likely be useful and which likely won't be. Some things I learned when reading:

YESFINRISK/NOFINRISK - Self reported risk tolerance derived from survey  
STOCKS — direct stock ownership  
STMUTF — stock mutual funds  
EQUITY — total equity holdings  
RETQLIQ — retirement account value  

Further, we can construct our own risk tolerance metric using some of these other:  
EQUITY, DEQ, RETEQ — equity holdings (total, direct, retirement)
STOCKS, HSTOCKS — stock ownership (value and indicator)
LEVRATIO — leverage ratio (debt/assets) — interesting risk signal
DEBT2INC — debt-to-income ratio

Portfolio composition signals:

STMUTF, TFBMUTF, GBMUTF, OBMUTF — mutual fund types
BOND, RETQLIQ — fixed income and retirement liquidity  


Maybe look for things like life insurance to show lower risk proxies etc. 

In [ ]:
print(data['YESFINRISK'].value_counts())
print(data['NOFINRISK'].value_counts())

Alright, financial risk would likely be highly imbalanced (~5% of households are explicitly willing to take financial risk and ~32% are unwilling). The rest are likely our moderate risk group but this is still too imbalanced. We can potentially calculate financial risk tolerance by looking how much of families assets are invested in equities. Let's take a look at what this would look like. 

In [ ]:
print(data['YY1'].value_counts())
print(data['Y1'].value_counts())

Alright, it is important to note that YY1 is the family identifier and that Y1 is a concatinated family identifier and the imputation. We will need to filter it so that we only take one of the five imputation forms. 

In [ ]:
df = data.copy()
df['imputation'] = df['Y1'].astype(str).str[-1]
df = df[df['imputation'] == '1']
print(f"Dataframe Shape after filtering for first imputation: {df.shape}")
df['equity_alloc'] = df['EQUITY'] / df['ASSET']
df['equity_alloc'] = df['equity_alloc'].clip(0, 1)
df['equity_alloc'].describe()

This is looking right and very interesting. We were expecting ~4600 families. We have 4574. An average equity investment is 14% of total assets with a low being 0 and high being 99.5%. Let's see what the age breakdown is.

In [ ]:
age = df['AGE'].value_counts().reset_index()
age.columns = ['AGE', 'count']

age_dist = alt.Chart(age).mark_bar(color='black').encode(
    x=alt.X('AGE:O'),
    y=alt.Y('count:Q', axis = alt.Axis(grid=False))
).properties(width=900, height=500, title="Age Distribution of Survey Respondents").configure_view(strokeWidth=0)

age_dist

Let's see if ASSETS are correlated with AGE.

In [ ]:
age_asset = alt.Chart(df).mark_point(filled=True, size=100).encode(
    x=alt.X('AGE:Q', axis=alt.Axis(grid=False)),
    y=alt.Y('ASSET:Q', axis = alt.Axis(grid=False)),
    color=alt.Color('equity_alloc:Q', scale=alt.Scale(scheme='viridis'), legend=alt.Legend(title="Equity Allocation"))
    ).configure_view(strokeWidth=0).properties(width=900,height=500,title="Age vs Asset with Equity Allocation as Color")

age_asset

This looks like there are a significant number of families that have over $200 Million in assets. This is going to require us to investigate a little further. It turns out that the SCF actually oversamples wealthy families in order to get a broader amount of data. It may be a good idea to use log scales as its common to have a right skew in financial data. Let's see what that looks like.

In [ ]:
age_asset_log = alt.Chart(df).mark_point(filled=True, size=100).encode(
    x=alt.X('AGE:Q', axis=alt.Axis(grid=False)),
    y=alt.Y('ASSET:Q', axis = alt.Axis(grid=False), scale=alt.Scale(type='log')),
    color=alt.Color('equity_alloc:Q', scale=alt.Scale(scheme='viridis'), legend=alt.Legend(title="Equity Allocation"))
    ).configure_view(strokeWidth=0).properties(width=900,height=500,title="Age vs Asset with Equity Allocation as Color")

age_asset_log

In [ ]:
#Let's look at equity to age
df['STOCKS'] = df['STOCKS'] + 1  # Clip equity values to a minimum of 1 to avoid log(0) issues
equity_age = alt.Chart(df).mark_point(filled=True, size=100).encode(
    x=alt.X('AGE:Q', axis=alt.Axis(grid=False)),
    y=alt.Y('STOCKS:Q', axis = alt.Axis(grid=False), scale=alt.Scale(type='log')),
    color=alt.Color('equity_alloc:Q', scale=alt.Scale(scheme='viridis'), legend=alt.Legend(title="Equity Allocation"))
    ).configure_view(strokeWidth=0).properties(width=900,height=500,title="Age vs Equity with Equity Allocation as Color")
equity_age

Just for interest, let's see what the wealth for people my age are.

In [ ]:
my_age = df[df['AGE']==40]
wealth = alt.Chart(my_age).mark_point(filled=True, size=100).encode(
    x=alt.X('STOCKS:Q', axis=alt.Axis(grid=False), scale=alt.Scale(type='log')),
    y=alt.Y('ASSET:Q', axis = alt.Axis(grid=False), scale=alt.Scale(type='log')),
    color=alt.Color('equity_alloc:Q', scale=alt.Scale(scheme='viridis'), legend=alt.Legend(title="Equity Allocation"))
).configure_view(strokeWidth=0).properties(width=900,height=500,title="Equity vs Wealth for 40 Year Olds with Equity Allocation as Color")
wealth

In [ ]:
my_age['ASSET'].describe()

The median (more reliable when talking about with) 40 year old had a net worth of $291,000. The mean value was heavily right skewed ($1.2 million) because a few wealthy individuals. For example the top 25% have over $10 million dollars.

Moving on from this, we can see that equities really don't make up a huge amount of the net worth of these individuals. THis makes me think that a lot of it may be tied up in realestate. Let's look at what different columns may be related to real estate:  
HOUSES — primary residence value  
ORESRE — value of other residential real estate (rental properties, vacation homes)  
NNRESRE — non-residential real estate (commercial property)  
MORTPAY / MORT1 — mortgage balances against those  


In [ ]:
df['HOUSES'] = df['HOUSES'] + 1  # Clip house values to a minimum of 1 to avoid log(0) issues
asset_res = alt.Chart(df).mark_point(filled=True, size=100).encode(
    x=alt.X('ASSET:Q', axis=alt.Axis(grid=False), scale=alt.Scale(type='log')),
    y=alt.Y('HOUSES:Q', axis = alt.Axis(grid=False), scale=alt.Scale(type='log')),
    color=alt.Color('equity_alloc:Q', scale=alt.Scale(scheme='viridis'), legend=alt.Legend(title="Equity Allocation"))
).configure_view(strokeWidth=0).properties(width=900,height=500,title="Asset vs Residual Income with Equity Allocation as Color")
asset_res

In [ ]:
my_age['HOUSES'] = my_age['HOUSES'] + 1  # Clip house values to a minimum of 1 to avoid log(0) issues
asset_res = alt.Chart(my_age).mark_point(filled=True, size=100).encode(
    x=alt.X('ASSET:Q', axis=alt.Axis(grid=False), scale=alt.Scale(type='log')),
    y=alt.Y('HOUSES:Q', axis = alt.Axis(grid=False), scale=alt.Scale(type='log')),
    color=alt.Color('equity_alloc:Q', scale=alt.Scale(scheme='viridis'), legend=alt.Legend(title="Equity Allocation"))
).configure_view(strokeWidth=0).properties(width=900,height=500,title="Asset vs Residual Income with Equity Allocation as Color")
asset_res